# Project Completion Local + Vertex Notebook

This notebook keeps the working path small: data-processing SQL is imported from `project_completion/data_processing.py`, the trainer can run locally from the notebook, and the same package can be submitted as a Vertex AI Custom Job. CI/CD files are unchanged. Environment values are read from shell variables or `.env`.

## Environment Setup

Run this once in a fresh local or Workbench environment.

In [1]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-local.txt"])

# Required before Vertex submit or GCS package upload:
#   gcloud auth application-default login
#   gcloud auth login
#   gcloud config set project $PROJECT_ID

import os
from pathlib import Path

env_candidates = [Path(".env"), Path("env"), Path("env.txt"), Path(".env.txt")]
env_file = next((path for path in env_candidates if path.exists()), None)

if env_file:
    print("Loading environment from:", env_file)
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
else:
    print("No env file found. Expected one of:", [str(path) for path in env_candidates])

required_env = ["PROJECT_ID", "REGION", "BUCKET"]
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    print("Missing env vars:", missing_env)
    print("Upload env.example or export these variables before running Vertex/GCS cells.")
else:
    print("Environment ready:", {name: os.getenv(name) for name in required_env})


Loading environment from: env.txt
Environment ready: {'PROJECT_ID': 'qwiklabs-asl-03-7c1aaee9a503', 'REGION': 'us-central1', 'BUCKET': 'qwiklabs-asl-03-7c1aaee9a503'}


In [2]:
import glob
import math
import os
import subprocess
from datetime import datetime

from project_completion.data_processing import (
    DataProcessingConfig,
    build_export_jobs,
    build_feature_sql,
    build_preview_query,
    build_split_sql,
)

## Configuration

In [3]:
def get_gcloud_project(default="qwiklabs-asl-03-7c1aaee9a503"):
    try:
        value = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
        return value or default
    except Exception:
        return default


PROJECT_ID = os.getenv("PROJECT_ID", get_gcloud_project())
REGION = os.getenv("REGION", "us-central1")
BUCKET = os.getenv("BUCKET", PROJECT_ID)
BUCKET_URI = f"gs://{BUCKET}"

config = DataProcessingConfig(
    project_id=PROJECT_ID,
    dataset_id=os.getenv("DATASET_ID", "jindong_lin"),
    table_id=os.getenv("TABLE_ID", "project_data"),
    bucket_uri=os.getenv("DATA_GCS_BASE_URI", f"{BUCKET_URI}/jindong_lin/data"),
)

TRAIN_DATA_PATH = f"{config.data_gcs_prefix}/train/project-train-*"
EVAL_DATA_PATH = f"{config.data_gcs_prefix}/valid/project-valid-*"
SCHEMA_PATH = os.getenv("SCHEMA_PATH", getattr(config, "schema_gcs_path", f"{config.data_gcs_prefix}/schema/feature_schema.json"))
LOCAL_OUTPUT_DIR = f"{BUCKET_URI}/project_completion/local_gbdt_test"

print("PROJECT_ID:", PROJECT_ID)
print("REGION:", REGION)
print("SOURCE_TABLE:", config.source_table)
print("TRAIN_DATA_PATH:", TRAIN_DATA_PATH)
print("EVAL_DATA_PATH:", EVAL_DATA_PATH)
print("SCHEMA_PATH:", SCHEMA_PATH)
print("LOCAL_OUTPUT_DIR:", LOCAL_OUTPUT_DIR)


PROJECT_ID: qwiklabs-asl-03-7c1aaee9a503
REGION: us-central1
SOURCE_TABLE: qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_dataform_features_mask_0615
TRAIN_DATA_PATH: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*
EVAL_DATA_PATH: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*
SCHEMA_PATH: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/schema/feature_schema.json
LOCAL_OUTPUT_DIR: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test


## Data Processing Smoke Test

In [4]:
preview_query = build_preview_query(config.source_table)
feature_sql = build_feature_sql(config.source_table, config.feature_table)
split_sql = build_split_sql(config.feature_table, config.train_table, config.valid_table, config.test_table)
export_jobs = build_export_jobs(config)

assert f"FROM `{config.source_table}`" in preview_query
assert "SELECT *" in feature_sql
assert "days_to_S90 IS NOT NULL" in feature_sql
assert "days_to_S90 >= 0" in feature_sql
#assert "so_nr" in split_sql
#assert "projekt_id" in split_sql
#assert "FARM_FINGERPRINT" in split_sql
assert export_jobs[0][1].endswith("/train/project-train-*.csv")
assert export_jobs[1][1].endswith("/valid/project-valid-*.csv")
assert config.schema_gcs_path.endswith("/schema/feature_schema.json")

print("Data-processing helper smoke test passed.")


Data-processing helper smoke test passed.


## Optional Data Processing

Keep `RUN_DATA_PROCESSING = False` for normal training runs. Set it to `True` only for the first run or when the source BigQuery table has changed and the GCS CSV files must be regenerated.

In [5]:
RUN_DATA_PROCESSING = os.getenv("RUN_DATA_PROCESSING", "False").lower() == "true"

if RUN_DATA_PROCESSING:
    from google.cloud import bigquery, storage
    from project_completion.data_processing import export_tables_to_gcs, run_bigquery_data_processing

    bq_client = bigquery.Client(project=PROJECT_ID)
    storage_client = storage.Client(project=PROJECT_ID)
    bq_location = os.getenv("BQ_LOCATION")
    if not bq_location:
        dataset_ref = bigquery.DatasetReference(PROJECT_ID, config.dataset_id)
        bq_location = bq_client.get_dataset(dataset_ref).location

    row_counts = run_bigquery_data_processing(bq_client, config, location=bq_location)
    export_tables_to_gcs(
        bq_client,
        config,
        location=bq_location,
        storage_client=storage_client,
        clean_existing=True,
    )
    print("Regenerated CSV files from BigQuery.")
    print(row_counts)
else:
    print("Skip data processing. Reusing existing CSV files in GCS:")
    print("  train:", TRAIN_DATA_PATH)
    print("  valid:", EVAL_DATA_PATH)


Skip data processing. Reusing existing CSV files in GCS:
  train: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*
  valid: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*


In [6]:
os.getenv("LOCAL_EPOCHS")

## Run the trainer locally from the notebook

In [10]:
def _env_int(name, default):
    return int(os.getenv(name, str(default)))


def _env_float(name, default):
    return float(os.getenv(name, str(default)))


N_ESTIMATORS = _env_int("N_ESTIMATORS", 500)
MAX_DEPTH = _env_int("MAX_DEPTH", 8)
GBDT_LEARNING_RATE = _env_float("GBDT_LEARNING_RATE", 0.1)
RANDOM_STATE = _env_int("RANDOM_STATE", 42)
LABEL_SCALE = _env_float("LABEL_SCALE", 1.0)

print("GBDT n_estimators:", N_ESTIMATORS)
print("GBDT max_depth:", MAX_DEPTH)
print("GBDT learning_rate:", GBDT_LEARNING_RATE)
print("Schema path:", SCHEMA_PATH)

local_train_cmd = [
    "python", "-m", "trainer.task",
    "--train_data_path", TRAIN_DATA_PATH,
    "--eval_data_path", EVAL_DATA_PATH,
    "--schema_path", SCHEMA_PATH,
    "--output_dir", LOCAL_OUTPUT_DIR,
    "--n_estimators", str(N_ESTIMATORS),
    "--max_depth", str(MAX_DEPTH),
    "--learning_rate", str(GBDT_LEARNING_RATE),
    "--random_state", str(RANDOM_STATE),
    "--label_scale", str(LABEL_SCALE),
]

env = os.environ.copy()
env.setdefault("CUDA_VISIBLE_DEVICES", "-1")
env.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

subprocess.run(local_train_cmd, cwd="project_completion", env=env, check=True)


GBDT n_estimators: 100
GBDT max_depth: 5
GBDT learning_rate: 0.05
Schema path: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/schema/feature_schema.json


2026-06-17 10:05:46.610806: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-17 10:05:46.641543: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-17 10:05:46.650567: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


{
  "eval_loss_mse": 4773.085414849801,
  "eval_mae_days": 40.29882741777848,
  "eval_r2": 0.9780974890205792,
  "eval_rmse_days": 69.0875199645334,
  "eval_rows": 8397,
  "feature_importance_by_original_feature_path": "gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test/feature_importance_by_original_feature.csv",
  "feature_importance_encoded_path": "gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test/feature_importance_encoded.csv",
  "feature_schema": {
    "categorical_features": [
      "zustaendige_region",
      "ziel_des_projekts",
      "area_gu",
      "dim_squads_mac"
    ],
    "csv_columns": [
      "so_nr",
      "projekt_id",
      "zustaendige_region",
      "ziel_des_projekts",
      "area_gu",
      "dim_squads_mac",
      "days_to_S90",
      "data_split",
      "missing_stx44",
      "days_stx44_from_stx30",
      "missing_stx51a",
      "days_stx51a_from_stx30",
      "missing_stx52",
      "days_stx52_from_stx30",
      "missing_

CompletedProcess(args=['python', '-m', 'trainer.task', '--train_data_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*', '--eval_data_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*', '--schema_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/schema/feature_schema.json', '--output_dir', 'gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test', '--n_estimators', '100', '--max_depth', '5', '--learning_rate', '0.05', '--random_state', '42', '--label_scale', '1.0'], returncode=0)

## View Local Training with TensorBoard

Run this cell after local training finishes to inspect TensorBoard logs written under `LOCAL_OUTPUT_DIR/tensorboard`.


In [9]:
# TensorBoard for the local trainer output.
# If the extension is already loaded, Jupyter may print a harmless message.
#!kill 28504
%load_ext tensorboard

TENSORBOARD_LOGDIR = f"{LOCAL_OUTPUT_DIR.rstrip('/')}/tensorboard"
print("TensorBoard logdir:", TENSORBOARD_LOGDIR)
%tensorboard --logdir {TENSORBOARD_LOGDIR} --port 8000


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
TensorBoard logdir: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test/tensorboard


Reusing TensorBoard on port 8000 (pid 25624), started 0:11:41 ago. (Use '!kill 25624' to kill it.)

## Submit Custom Job using the Vertex AI SDK

This section creates or reuses a Vertex AI TensorBoard instance by default, gets the compute service account with `gcloud`, and submits the Custom Job with both `service_account` and `tensorboard`.


In [12]:
#%pip install -r requirements-local.txt

  Using cached google_cloud_aiplatform-1.69.0-py2.py3-none-any.whl.metadata (32 kB)
  Using cached google_cloud_bigquery-3.25.0-py2.py3-none-any.whl.metadata (8.9 kB)
  Using cached tensorflow-2.17.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached tensorboard-2.17.1-py3-none-any.whl.metadata (1.6 kB)
Using cached google_cloud_aiplatform-1.69.0-py2.py3-none-any.whl (5.3 MB)
Using cached google_cloud_bigquery-3.25.0-py2.py3-none-any.whl (239 kB)
Using cached tensorflow-2.17.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (601.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 96.1 MB/s  0:00:00
Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.0 MB/s  0:00:00
Using cached tensorboard-2.17.1-py3-none-any.whl (5.

In [7]:
from pathlib import Path
import os
import subprocess
import sys

RUN_VERTEX_CUSTOM_JOB = os.getenv("RUN_VERTEX_CUSTOM_JOB", "True").lower() == "true"

cmd = [sys.executable, "scripts/submit_training_job.py", "--env_file", "env.txt"]
if not RUN_VERTEX_CUSTOM_JOB:
    cmd.append("--no_submit")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /opt/micromamba/envs/jupyterlab/bin/python3 scripts/submit_training_job.py --env_file env.txt
running sdist
running egg_info
writing project_completion_trainer.egg-info/PKG-INFO
writing dependency_links to project_completion_trainer.egg-info/dependency_links.txt
writing requirements to project_completion_trainer.egg-info/requires.txt
writing top-level names to project_completion_trainer.egg-info/top_level.txt
reading manifest file 'project_completion_trainer.egg-info/SOURCES.txt'
writing manifest file 'project_completion_trainer.egg-info/SOURCES.txt'
running check
creating project_completion_trainer-0.1
creating project_completion_trainer-0.1/project_completion_trainer.egg-info
creating project_completion_trainer-0.1/trainer
copying files to project_completion_trainer-0.1...
copying setup.py -> project_completion_trainer-0.1
copying project_completion_trainer.egg-info/PKG-INFO -> project_completion_trainer-0.1/project_completion_trainer.egg-info
copying project_completion_trai


Copying file://project_completion/dist/project_completion_trainer-0.1.tar.gz to gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/packages/project_completion_trainer_20260617_100319.tar.gz
  
.


Local package: project_completion/dist/project_completion_trainer-0.1.tar.gz
PACKAGE_URI: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/packages/project_completion_trainer_20260617_100319.tar.gz
MODEL_OUTPUT_DIR: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/trained_gbdt_model_20260617_100319
TensorBoard: projects/192763454488/locations/us-central1/tensorboards/6279198756042702848
Service account: 192763454488-compute@developer.gserviceaccount.com
Training args: ['--train_data_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*.csv', '--eval_data_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*.csv', '--schema_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/schema/feature_schema.json', '--output_dir', 'gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/trained_gbdt_model_20260617_100319', '--n_estimators', '100', '--max_depth', '5', '--learnin

CompletedProcess(args=['/opt/micromamba/envs/jupyterlab/bin/python3', 'scripts/submit_training_job.py', '--env_file', 'env.txt'], returncode=0)

## Running analysis and predict



In [11]:
VERTEX_MODEL_DIR = os.getenv("VERTEX_MODEL_DIR", LOCAL_OUTPUT_DIR)
ACTIVE_MODEL_PATH = f"{VERTEX_MODEL_DIR.rstrip('/')}/model.joblib"

print("Using GBDT model:", ACTIVE_MODEL_PATH)


Using GBDT model: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test/model.joblib


In [13]:
import sklearn
print(sklearn.__version__)

1.9.0


In [12]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import r2_score

from project_completion.trainer import model as trainer_model

schema = trainer_model.load_feature_schema(SCHEMA_PATH)
model_path = ACTIVE_MODEL_PATH
print("Loading model from:", model_path)

if model_path.startswith("gs://"):
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        tf.io.gfile.copy(model_path, tmp.name, overwrite=True)
        trained_model = joblib.load(tmp.name)
else:
    trained_model = joblib.load(model_path)

TEST_DATA_PATH = f"{config.data_gcs_prefix}/test/project-test-*"
test_df = trainer_model.read_csv_dataset(TEST_DATA_PATH, schema)
x_test, y_test = trainer_model.split_features_label(test_df, schema, label_scale=LABEL_SCALE)
y_pred = trained_model.predict(x_test)

pred_df = pd.DataFrame({
    "actual_days_to_S90": y_test * LABEL_SCALE,
    "predicted_days_to_S90": y_pred * LABEL_SCALE,
})
pred_df["error_days"] = pred_df["predicted_days_to_S90"] - pred_df["actual_days_to_S90"]

mae = np.mean(np.abs(pred_df["error_days"]))
rmse = np.sqrt(np.mean(np.square(pred_df["error_days"])))
r2 = r2_score(pred_df["actual_days_to_S90"], pred_df["predicted_days_to_S90"])

print(f"Rows: {len(pred_df):,}")
print(f"MAE days: {mae:.2f}")
print(f"RMSE days: {rmse:.2f}")
print(f"R2: {r2:.4f}")

pred_df.head()


Loading model from: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_gbdt_test/model.joblib


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.3.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.3.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.3.2 when us

ModuleNotFoundError: No module named 'sklearn.ensemble._gb_losses'

In [ ]:
# Feature importance by original input feature

from io import StringIO

model_output_dir = model_path.rsplit("/", 1)[0]
feature_importance_path = f"{model_output_dir}/feature_importance_by_original_feature.csv"

if tf.io.gfile.exists(feature_importance_path):
    with tf.io.gfile.GFile(feature_importance_path, "r") as f:
        feature_importance_df = pd.read_csv(f)
else:
    preprocess = trained_model.named_steps["preprocess"]
    regressor = trained_model.named_steps["model"]
    encoded_importance_df = pd.DataFrame({
        "encoded_feature": preprocess.get_feature_names_out(),
        "importance": regressor.feature_importances_,
    })

    def original_feature_name(encoded_name):
        name = encoded_name.split("__", 1)[-1]
        for feature_name in sorted(schema["categorical_features"], key=len, reverse=True):
            if name == feature_name or name.startswith(f"{feature_name}_"):
                return feature_name
        for feature_name in schema["numeric_features"]:
            if name == feature_name:
                return feature_name
        return name

    encoded_importance_df["original_feature"] = encoded_importance_df["encoded_feature"].map(original_feature_name)
    feature_importance_df = (
        encoded_importance_df.groupby("original_feature", as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

feature_importance_df.head(20)


In [ ]:
# Plot top feature importance

top_n = 20
plot_df = feature_importance_df.head(top_n).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["original_feature"], plot_df["importance"])
plt.xlabel("GBDT feature importance")
plt.title(f"Top {top_n} Feature Importance by Original Feature")
plt.grid(axis="x", alpha=0.2)
plt.show()


In [ ]:
# 1. Actual vs predicted distribution
plt.figure(figsize=(10, 5))
plt.hist(pred_df["actual_days_to_S90"], bins=80, density=True, alpha=0.45, label="Actual")
plt.hist(pred_df["predicted_days_to_S90"], bins=80, density=True, alpha=0.45, label="Predicted")
plt.xlabel("days_to_S90")
plt.ylabel("Density")
plt.title("Actual vs Predicted Distribution")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# 2. Predicted vs actual scatter
plt.figure(figsize=(6, 6))
plt.scatter(pred_df["actual_days_to_S90"], pred_df["predicted_days_to_S90"], alpha=0.25, s=8)
max_v = max(pred_df["actual_days_to_S90"].max(), pred_df["predicted_days_to_S90"].max())
plt.plot([0, max_v], [0, max_v], "r--", label="Perfect prediction")
plt.xlabel("Actual days_to_S90")
plt.ylabel("Predicted days_to_S90")
plt.title("Predicted vs Actual")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# 3. Error distribution
plt.figure(figsize=(10, 5))
plt.hist(pred_df["error_days"], bins=80, density=True, alpha=0.7)
plt.axvline(0, color="red", linestyle="--")
plt.xlabel("Prediction error days: predicted - actual")
plt.ylabel("Density")
plt.title("Prediction Error Distribution")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
predictions_path = f"{LOCAL_OUTPUT_DIR.rstrip('/')}/predictions/test_predictions.csv"
tf.io.gfile.makedirs(f"{LOCAL_OUTPUT_DIR.rstrip('/')}/predictions")

with tf.io.gfile.GFile(predictions_path, "w") as f:
    pred_df.to_csv(f, index=False)

print("Saved predictions to:", predictions_path)


In [ ]:
from google.cloud import bigquery
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from project_completion.trainer import model as trainer_model

schema = trainer_model.load_feature_schema(SCHEMA_PATH)
model_path = ACTIVE_MODEL_PATH
print("Loading model from:", model_path)

if model_path.startswith("gs://"):
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        tf.io.gfile.copy(model_path, tmp.name, overwrite=True)
        trained_model = joblib.load(tmp.name)
else:
    trained_model = joblib.load(model_path)

bq_client = bigquery.Client(project=PROJECT_ID)

SCORING_TABLE = "qwiklabs-asl-03-7c1aaee9a503.jindong_lin.frozenlist-2026-forecast"

input_columns = schema["categorical_features"] + schema["numeric_features"]
meta_columns = [
    "so_nr",
    "projekt_id",
    "ist_stx30",
    "ist_stx90",
]

select_columns = meta_columns + input_columns
select_sql = ",".join([f"`{c}`" for c in select_columns])

score_sql = f"""
SELECT
  {select_sql}
FROM `{SCORING_TABLE}`
WHERE ist_stx30 IS NOT NULL
"""

score_df = bq_client.query(score_sql).to_dataframe()
print("Rows to predict:", len(score_df))
score_df.head()


In [ ]:
score_features = score_df[input_columns].copy()
for col in schema["categorical_features"]:
    score_features[col] = score_features[col].fillna("UNKNOWN").astype(str)
for col in schema["numeric_features"]:
    score_features[col] = pd.to_numeric(score_features[col], errors="coerce")

predicted_days = trained_model.predict(score_features) * LABEL_SCALE

prediction_df = score_df[meta_columns].copy()
prediction_df["predicted_days_to_S90"] = predicted_days
prediction_df["predicted_months_to_S90"] = prediction_df["predicted_days_to_S90"] / 30.4375
prediction_df["predicted_quarters_to_S90"] = prediction_df["predicted_days_to_S90"] / 91.3125

prediction_df["ist_stx30"] = pd.to_datetime(prediction_df["ist_stx30"])
prediction_df["ist_stx90"] = pd.to_datetime(prediction_df["ist_stx90"])

prediction_df["predicted_s90_date"] = prediction_df["ist_stx30"] + pd.to_timedelta(
    prediction_df["predicted_days_to_S90"].round().astype(int),
    unit="D",
)

prediction_df["predicted_s90_month"] = prediction_df["predicted_s90_date"].dt.to_period("M").astype(str)
prediction_df["predicted_s90_quarter"] = prediction_df["predicted_s90_date"].dt.to_period("Q").astype(str)

prediction_df["actual_s90_month"] = prediction_df["ist_stx90"].dt.to_period("M").astype(str)
prediction_df["actual_s90_quarter"] = prediction_df["ist_stx90"].dt.to_period("Q").astype(str)

prediction_df.head(20)


In [ ]:
completed_df = prediction_df[prediction_df["ist_stx90"].notna()].copy()

completed_df["actual_days_to_S90"] = (
    completed_df["ist_stx90"] - completed_df["ist_stx30"]
).dt.days

completed_df["error_days"] = completed_df["predicted_days_to_S90"] - completed_df["actual_days_to_S90"]

completed_summary = pd.DataFrame({
    "metric": ["mae_days", "rmse_days", "r2", "bias_days"],
    "value": [
        np.mean(np.abs(completed_df["error_days"])),
        np.sqrt(np.mean(np.square(completed_df["error_days"]))),
        r2_score(completed_df["actual_days_to_S90"], completed_df["predicted_days_to_S90"]),
        np.mean(completed_df["error_days"]),
    ],
})

completed_summary

In [ ]:
OUTPUT_TABLE = "qwiklabs-asl-03-7c1aaee9a503.jindong_lin.frozenlist_2026_forecast_predictions"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
)

load_job = bq_client.load_table_from_dataframe(
    prediction_df,
    OUTPUT_TABLE,
    job_config=job_config,
)
load_job.result()

In [ ]:
print("Saved BigQuery table:", OUTPUT_TABLE)


In [ ]:
compare_sql = """
WITH test_keys AS (
  SELECT
    CAST(so_nr AS STRING) AS so_nr,
    CAST(projekt_id AS STRING) AS projekt_id,
    days_to_S90
  FROM `qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_test`
),
score_completed AS (
  SELECT
    CAST(so_nr AS STRING) AS so_nr,
    CAST(projekt_id AS STRING) AS projekt_id,
    DATE_DIFF(DATE(ist_stx90), DATE(ist_stx30), DAY) AS actual_days_to_S90
  FROM `qwiklabs-asl-03-7c1aaee9a503.jindong_lin.frozenlist-2026-forecast`
  WHERE ist_stx30 IS NOT NULL
    AND ist_stx90 IS NOT NULL
)
SELECT
  COUNT(*) AS joined_rows,
  COUNTIF(t.so_nr IS NULL) AS only_in_score_completed,
  COUNTIF(s.so_nr IS NULL) AS only_in_test,
  AVG(ABS(t.days_to_S90 - s.actual_days_to_S90)) AS avg_label_diff
FROM test_keys t
FULL OUTER JOIN score_completed s
USING (so_nr, projekt_id)
"""

bq_client.query(compare_sql).to_dataframe()
